# 1. Setup: Packages and Global Parameters


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import requests

URL = "https://incidentdatabase.ai/api/graphql"

introspection_query = """
{
  __type(name: "Incident") {
    name
    fields {
      name
      type { name kind ofType { name kind } }
    }
  }
}
"""

# The AIID GraphQL endpoint rejects POST requests whose Origin/Referer header
# isn't https://incidentdatabase.ai or https://staging-aiid.netlify.app
# (see site/gatsby-site/netlify/functions/graphql.ts in responsible-ai-collaborative/aiid).
# That check only applies to POST — GET requests bypass it, so we send the
# query as a URL parameter instead.
r = requests.get(URL, params={"query": introspection_query})
print(r.json())

In [ ]:
import glob
import json
import os

import pandas as pd

# Folder in Google Drive containing the batch JSON files to combine.
BATCH_DIR = "/content/drive/MyDrive/phd/P4/Data"
BATCH_PATTERN = "batch*.json"
OUTPUT_CSV = os.path.join(BATCH_DIR, "aiid_full_incidents_processed.csv")

batch_paths = sorted(glob.glob(os.path.join(BATCH_DIR, BATCH_PATTERN)))
print(f"Found {len(batch_paths)} batch file(s)")

combined = []

for path in batch_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Each batch file may be a JSON array of records, or a single object.
    if isinstance(data, list):
        combined.extend(data)
    else:
        combined.append(data)

    print(f"  {os.path.basename(path)}: {len(data) if isinstance(data, list) else 1} record(s)")

print(f"Total combined records: {len(combined)}")

# Flatten nested fields (e.g. reports, entities) into dot-separated columns;
# any remaining list/dict values are JSON-stringified so they stay CSV-safe.
df = pd.json_normalize(combined)
for col in df.columns:
    if df[col].apply(lambda v: isinstance(v, (list, dict))).any():
        df[col] = df[col].apply(lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v)

df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(df)} rows to {OUTPUT_CSV}")

# 2. Descriptive Analysis


In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/phd/P4/Data/aiid_full_incidents_processed.csv"

df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print("Columns and dtypes:")
print(df.dtypes)

In [ ]:
# Missing values per column
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0]

In [ ]:
# Summary statistics for numeric columns
df.describe(include="number").T

In [ ]:
# Value counts for object/categorical columns with manageable cardinality
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    n_unique = df[col].nunique(dropna=True)
    if 1 < n_unique <= 30:
        print(f"--- {col} ({n_unique} unique values) ---")
        print(df[col].value_counts(dropna=False).head(10))
        print()

In [ ]:
# Incidents over time, if a date-like column is present (e.g. "date")
date_cols = [c for c in df.columns if "date" in c.lower()]

if date_cols:
    date_col = date_cols[0]
    dates = pd.to_datetime(df[date_col], errors="coerce")
    print(f"Using date column: {date_col}")
    print(f"Range: {dates.min()} to {dates.max()}")
    print()
    print("Incidents per year:")
    print(dates.dt.year.value_counts().sort_index())
else:
    print("No date-like column found.")